In [1]:
import os
import numpy as np
import pandas as pd
import pandas_market_calendars as mcal
import datetime as dt
from data_processor import DataReader, DataPrep
from scipy import stats

In [2]:
nyse = mcal.get_calendar('NYSE')
# Get holidays
holidays = nyse.holidays().holidays

### EDA

In [3]:
daily_data_path = r'data/daily_data'
intraday_data_path = r'data/intraday_data'

In [4]:
reader = DataReader()
intraday_df = reader.read_intraday_data(intraday_data_path)
daily_df = reader.read_daily_data(daily_data_path)
intraday_df.dropna(subset= 'CumReturnResid', inplace=True)

In [89]:
daily_df.eval('Range = abs(Close - Open)', inplace= True)

In [94]:
daily_df.query('SharesAdjFactor > 1')

,Date,Id,SYMBOL,MIC,FREE_FLOAT_PERCENTAGE,EST_VOL,MDV_63,Open,High,Low,Close,Volume,PxAdjFactor,SharesAdjFactor,n_split,Stock_Split,Dividend,Range
172,2010-01-04,BBG000BF8TF5,CDE,XNYS,99.8983,0.25942,74229670.0,18.63,19.22,18.59,18.74,3687419.0,0.100000,10.000000,10.000000,True,False,0.11
203,2010-01-04,BBG000BBDZG3,AIG,XNYS,16.7945,0.31906,490784030.0,30.53,30.54,29.41,29.89,7749986.0,0.051011,20.000000,19.603744,True,False,0.64
217,2010-01-04,BBG000H89QJ6,TWC,XNYS,99.9796,0.12911,86349790.0,42.05,42.67,41.78,42.29,2918680.0,0.772785,3.000003,1.294021,True,False,0.24
385,2010-01-04,BBG000F0PF65,TWX,XNYS,99.8740,0.14605,198239600.0,29.22,29.52,29.22,29.42,6736346.0,0.375511,3.000003,2.663034,True,False,0.20
167,2010-01-05,BBG000BF8TF5,CDE,XNYS,99.8983,0.27412,73422590.0,19.09,19.34,18.70,19.05,2431020.0,0.100000,10.000000,10.000000,True,False,0.04
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
400,2014-12-31,BBG000H89QJ6,TWC,XNYS,99.8731,0.17421,291105440.0,155.45,155.95,151.95,152.06,1224555.0,0.882723,3.000003,1.132858,True,False,3.39
407,2014-12-31,BBG000F0PF65,TWX,XNYS,94.3884,0.12042,327625250.0,86.20,86.71,85.36,85.42,2531978.0,0.441660,3.000003,2.264186,True,False,0.78
420,2014-12-31,BBG000NDYB67,GM,XNYS,84.7454,0.13035,411919140.0,35.24,35.45,34.89,34.91,12231966.0,0.010353,100.000000,96.585724,True,False,0.33
438,2014-12-31,BBG000BBDZG3,AIG,XNYS,99.9684,0.06884,407008960.0,56.73,56.75,55.97,56.01,5053976.0,0.061710,20.000000,16.204834,True,False,0.72


In [92]:
daily_df.groupby('Id').apply(lambda x: x.Range.shift().corr(x.EST_VOL)).describe()

c:\Users\klin2\AppData\Local\Programs\Python\Python311\Lib\site-packages\numpy\lib\function_base.py:2889: RuntimeWarning: Degrees of freedom <= 0 for slice
  c = cov(x, y, rowvar, dtype=dtype)
c:\Users\klin2\AppData\Local\Programs\Python\Python311\Lib\site-packages\numpy\lib\function_base.py:2748: RuntimeWarning: divide by zero encountered in divide
  c *= np.true_divide(1, fact)


count    861.000000
mean       0.116249
std        0.163737
min       -0.994682
25%        0.043541
50%        0.133573
75%        0.198632
max        1.000000
dtype: float64

0       6620.630484
1       6640.253308
2       8711.272008
3       6643.337264
4       6663.738590
           ...     
495    37191.353834
496    35682.356985
497    39865.962926
498    47528.344175
499    71367.013389
Name: MDV_63, Length: 629000, dtype: float64

In [5]:
def query_id(df):
    
    df = df[df.CumReturnResid.isna()]
    df.drop_duplicates('Date', inplace= True)
    df['Diff'] = df.Date.diff().dt.days
    return df[['Date', 'Diff']].sort_values('Diff')

In [6]:
test_df = intraday_df.groupby('Id').apply(query_id)

In [7]:
test_df = intraday_df.query('Id == "BBG000BLM0V1"')#.drop_duplicates('Date').reset_index(drop= True).iloc[680:690]#query('Date == @query_date').copy()
test_df[test_df.CumReturnResid.isna()].drop_duplicates('Date').reset_index(drop= True).iloc[60:66]#query('Date == @query_date').copy()

,Date,Time,Id,CumReturnResid,CumReturnRaw,CumVolume


In [8]:
data_prep = DataPrep(intraday_df, daily_df)

In [25]:
target_df = data_prep.get_target(clip_MAD=True, normalize= True)

In [26]:
target_df

,Id,Date,y,EST_VOL,MAD
0,BBG000B9WH86,2010-01-04,-0.032613,0.17017,0.006565
1,BBG000B9WJ73,2010-01-04,0.165352,0.15981,0.006565
2,BBG000B9XRY4,2010-01-04,-0.025006,0.17121,0.006565
3,BBG000B9XYV2,2010-01-04,-0.058475,0.10602,0.006565
4,BBG000B9YJ35,2010-01-04,-0.074573,0.15031,0.006565
...,...,...,...,...,...
625552,BBG002S5ZRF9,2014-12-30,-0.002446,0.30894,0.003953
625553,BBG002W96FT9,2014-12-30,0.042955,0.14773,0.003953
625554,BBG0039320N9,2014-12-30,-0.054349,0.22163,0.003953
625555,BBG005P7Q881,2014-12-30,-0.011752,0.13788,0.003953


### ID with different names

### Features Prep

In [95]:
daily_df

,Date,Id,SYMBOL,MIC,FREE_FLOAT_PERCENTAGE,EST_VOL,MDV_63,Open,High,Low,Close,Volume,PxAdjFactor,SharesAdjFactor,n_split,Stock_Split,Dividend,Range
0,2010-01-04,BBG000MQ1SN9,DVA,XNYS,99.6464,0.13074,4.383275e+07,59.12,60.05,59.09,59.92,955120.0,1.000000,1.000000,1.000000,False,False,0.80
1,2010-01-04,BBG000BBCQD7,SLM,XNYS,99.1117,0.31610,4.409296e+07,11.45,11.72,11.32,11.54,2566745.0,1.000000,1.000000,1.000000,False,False,0.09
2,2010-01-04,BBG000BBB3K1,RAI,XNYS,57.7643,0.11253,7.588626e+07,53.33,53.52,53.00,53.24,812905.0,1.160364,1.000000,0.861799,False,True,0.09
3,2010-01-04,BBG000BNMHS4,MAN,XNYS,99.3401,0.18568,4.413393e+07,54.94,56.58,54.78,56.41,1108030.0,1.034430,1.000000,0.966716,False,True,1.47
4,2010-01-04,BBG000C23PB0,SAI,XNYS,99.1347,0.11252,4.440541e+07,19.02,19.17,18.89,19.11,3500468.0,1.000000,1.000000,1.000000,False,False,0.09
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
495,2014-12-31,BBG000BPH459,MSFT,XNGS,91.5633,0.12146,1.383197e+09,46.73,47.44,46.45,46.45,21551092.0,1.188881,1.000000,0.841127,False,True,0.28
496,2014-12-31,BBG000GZQ728,XOM,XNYS,99.7689,0.09480,1.273231e+09,92.42,93.13,92.06,92.45,11326179.0,1.188282,1.000000,0.841551,False,True,0.03
497,2014-12-31,BBG000CKGBP2,GILD,XNGS,99.4121,0.20535,1.589295e+09,96.00,96.75,94.24,94.26,13851283.0,2.000000,0.500000,0.500000,True,False,1.74
498,2014-12-31,BBG000MM2P62,FB,XNGS,95.9988,0.17343,2.258944e+09,79.54,79.80,77.86,78.02,20004654.0,1.000000,1.000000,1.000000,False,False,1.52
